# Structural Consistency and Algorithm Timing on Standard Networks
This Notebook contains structural-consistency and timing experiments on six standard networks: Asia, Insurance, Alarm, Win95pts, Andes, and Munin. The consistency and timing experiments are executed independently.

In [1]:
from __future__ import annotations
import gc
import json
import random
import re
import time
from dataclasses import asdict, dataclass
from itertools import combinations
from pathlib import Path
from statistics import median
from typing import Hashable, Iterable, Mapping, Sequence
import networkx as nx
import pandas as pd

Node = Hashable
Edge = tuple[Node, Node]

## Experimental Input and Recording Utilities
The checks in this section are used to load the standard networks and prepare the experimental inputs.

In [2]:
# Check whether the input graph G is a valid simple directed acyclic graph (DAG).
def _validate_dag(G: nx.DiGraph) -> None:
    if not isinstance(G, nx.DiGraph):
        raise TypeError("G 必须是 networkx.DiGraph。")
    if G.is_multigraph():
        raise TypeError("G 必须是简单有向图，不能是 MultiDiGraph。")
    if not nx.is_directed_acyclic_graph(G):  
       # Use the NetworkX function to check whether the graph contains a cycle.
        raise ValueError("输入图 G 必须是 DAG。")

# Provide a sorting key for edges to ensure a stable order during serialization or comparison.
def _edge_sort_key(edge: Edge) -> tuple[str, str]:   # Accept an edge (source, target) and convert both endpoints to strings.
    return str(edge[0]), str(edge[1])

# Encode a node set as a compact JSON string for saving to Excel.
def _encode_nodes(nodes: Iterable[Node]) -> str:
    return json.dumps([str(node) for node in sorted(nodes, key=str)], ensure_ascii=False, separators=(",", ":"))

# Encode a topological order (node sequence) as a JSON string.
def _encode_order(order: Sequence[Node]) -> str:
    return json.dumps([str(node) for node in order], ensure_ascii=False, separators=(",", ":"))

# Encode a set of directed edges as a JSON string.
def _encode_edges(edges: Iterable[Edge]) -> str:
    ordered = sorted(edges, key=_edge_sort_key)
    return json.dumps([[str(source), str(target)] for source, target in ordered], ensure_ascii=False, separators=(",", ":"))

## Generate a valid topological order using randomized Kahn's algorithm, with a fixed seed to ensure reproducibility.
def random_topological_order(G: nx.DiGraph, seed: int) -> tuple[Node, ...]:
    _validate_dag(G)
    rng = random.Random(seed)   # Create the random-number generator.
    indegree = dict(G.in_degree())  # Record the current in-degree of each node.
    available = sorted([node for node, degree in indegree.items() if degree == 0], key=str)  # Initialize the list of zero-in-degree nodes.
    order: list[Node] = []
    while available:
        selected_index = rng.randrange(len(available))
        selected_node = available.pop(selected_index)
        order.append(selected_node)
        for child in sorted(G.successors(selected_node), key=str):
            indegree[child] -= 1
            if indegree[child] == 0:
                available.append(child)
        available.sort(key=str)
    if len(order) != G.number_of_nodes():
        raise RuntimeError("拓扑排序未覆盖 G 的全部节点。")
    return tuple(order)

# Sample n_sets distinct conditioning sets of size c_size without replacement.
def sample_random_condition_sets(G: nx.DiGraph, n_sets: int, c_size: int, seed: int) -> list[tuple[Node, ...]]:
    nodes = tuple(sorted(G.nodes(), key=str))   # Sort the node list by string representation to make sampling reproducible.
    if not 1 <= c_size < len(nodes):
        raise ValueError("必须满足 1 <= c_size < |V|。")
    rng = random.Random(seed)  # Create the random-number generator.
    selected: list[tuple[Node, ...]] = []
    seen: set[tuple[Node, ...]] = set()
    max_attempts = max(10_000, n_sets * 1_000)   # Set a maximum number of attempts to avoid an infinite loop.
    for _ in range(max_attempts):
        candidate = tuple(sorted(rng.sample(nodes, c_size), key=str))
        if candidate not in seen:
            seen.add(candidate)
            selected.append(candidate)
            if len(selected) == n_sets:
                return selected
    raise RuntimeError(f"只生成了 {len(selected)} 个不同条件集，目标为 {n_sets}。")

# Generate n_orders distinct random topological orders for a fixed network; each order corresponds to the seed seed_base + attempt.
def generate_fixed_random_topological_orders(G: nx.DiGraph, n_orders: int, seed_base: int, max_attempts: int = 10_000) -> list[dict[str, object]]:
    records: list[dict[str, object]] = []  # Store the generated results.
    seen: set[tuple[Node, ...]] = set()  
    for attempt in range(max_attempts):
        beta_seed = seed_base + attempt
        beta = random_topological_order(G, seed=beta_seed)
        if beta in seen:
            continue
        seen.add(beta)
        records.append({"order_id": len(records) + 1, "beta_seed": beta_seed, "beta": beta})
        if len(records) == n_orders:
            return records
    raise RuntimeError(f"{G.name or 'G'} 只生成了 {len(records)} 个不同拓扑序，目标为 {n_orders}。")

## FGCR Algorithm Implementation
Algorithm1 constructs $\alpha_C$、$\alpha_R$, and
$\alpha=(\alpha_C,\alpha_R)$. Algorithm2 performs covered-arc reversals and extracts
\$H_R=H[R]$；


In [3]:
# Algorithm result objects
@dataclass(frozen=True)
class Algorithm1Orders:
    beta: tuple[Node, ...]
    alpha_C: tuple[Node, ...]
    alpha_R: tuple[Node, ...]
    alpha: tuple[Node, ...]

@dataclass
class FGCRResult:
    H: nx.DiGraph
    H_R: nx.DiGraph
    orders: Algorithm1Orders

# FGCR Algorithm 1: (G, C, beta) -> alpha_C, alpha_R, alpha
def construct_algorithm1_orders(G: nx.DiGraph, C: Iterable[Node], beta: Sequence[Node]) -> Algorithm1Orders:
    C_set = set(C)
    beta_tuple = tuple(beta)
    # Traverse beta: nodes in C form alpha_C, while the remaining nodes form alpha_R.
    alpha_C = tuple(node for node in beta_tuple if node in C_set)
    alpha_R = tuple(node for node in beta_tuple if node not in C_set)
    return Algorithm1Orders(beta=beta_tuple, alpha_C=alpha_C, alpha_R=alpha_R, alpha=alpha_C + alpha_R)

# Cover the arc Y->Z and then reverse it to Z->Y.
def _cover_and_reverse(H: nx.DiGraph, Y: Node, Z: Node) -> None:
    pa_Y = set(H.predecessors(Y))  #获取 Y 和 Z 的父节点集合
    pa_Z = set(H.predecessors(Z))
    for X in pa_Y - pa_Z:
        H.add_edge(X, Z)
    for X in pa_Z - pa_Y - {Y}:
        H.add_edge(X, Y)
    H.remove_edge(Y, Z)
    H.add_edge(Z, Y)

# Algorithm 2: full-graph transformation
def _construct_full_H(G: nx.DiGraph, beta: Sequence[Node], alpha: Sequence[Node]) -> nx.DiGraph:
    beta_current = list(beta)
    alpha_list = list(alpha)
    beta_position = {node: index for index, node in enumerate(beta_current)}
    alpha_position = {node: index for index, node in enumerate(alpha_list)}
    H = G.copy()  # Copy the input graph.
    for Y in reversed(alpha_list):
        while beta_position[Y] < len(beta_current) - 1:
            current_position = beta_position[Y]
            Z = beta_current[current_position + 1]  # Z is the node immediately to the right of Y.
            if alpha_position[Z] > alpha_position[Y]:  # Z already lies to the right of Y in the target order alpha.
                break
            if H.has_edge(Y, Z):  # If Y->Z exists in H, perform the covered-arc reversal.
                _cover_and_reverse(H, Y, Z)
            beta_current[current_position] = Z  # Swap Y and Z in beta_current and update the position map.
            beta_current[current_position + 1] = Y
            beta_position[Z] = current_position
            beta_position[Y] = current_position + 1
    return H

# Untimed execution interface: run FGCR and return H_R as the final structural output.
def run_fgcr(G: nx.DiGraph, C: Iterable[Node], beta: Sequence[Node]) -> FGCRResult:   # Directly call Algorithms 1 and 2 without timing.
    orders = construct_algorithm1_orders(G=G, C=C, beta=beta)
    H = _construct_full_H(G=G, beta=orders.beta, alpha=orders.alpha)
    H_R = H.subgraph(orders.alpha_R).copy()
    return FGCRResult(H=H, H_R=H_R, orders=orders)

# Timed execution function; all elapsed times are measured in nanoseconds.
def _timed_fgcr_once(G: nx.DiGraph, C: Iterable[Node], beta: Sequence[Node]) -> tuple[FGCRResult, int, int]:
    # Timing interval for Algorithm 1.
    algorithm1_start = time.perf_counter_ns()
    orders = construct_algorithm1_orders(G=G, C=C, beta=beta)
    algorithm1_end = time.perf_counter_ns()
    # Timing interval for Algorithm 2.
    algorithm2_start = time.perf_counter_ns()
    H = _construct_full_H(G=G, beta=orders.beta, alpha=orders.alpha)
    H_R = H.subgraph(orders.alpha_R).copy()
    algorithm2_end = time.perf_counter_ns()
    result = FGCRResult(H=H, H_R=H_R, orders=orders)
    return (result, algorithm1_end - algorithm1_start, algorithm2_end - algorithm2_start)

# ROS Algorithm Implementation
ROS retains the algorithmic steps of removing outgoing edges from conditioned nodes, computing the ancestral closure, completing parent sets in reverse order, and constructing the final graph $G_R^\beta$

In [4]:
# Container for the results returned by the ROS algorithm.
@dataclass
class ROSResult:
    E_beta_R: tuple[Edge, ...]
    G_beta_R: nx.DiGraph

# Core ROS algorithm.
def _ros_algorithm_core(G: nx.DiGraph, beta: Sequence[Node], C: Iterable[Node]) -> tuple[list[Edge], nx.DiGraph]:
    C_set = set(C)
    beta_list = list(beta) # Convert the topological order beta to a list and record each node's position for edge orientation.
    beta_position = {node: index for index, node in enumerate(beta_list)}
    G_work = G.copy()
    # Construct G^C by removing all outgoing edges from conditioned nodes.
    for conditioned_node in C_set:
        G_work.remove_edges_from(list(G_work.out_edges(conditioned_node)))
     # Compute the ancestral closure.
    nodes_of_interest: set[Node] = set()
    for conditioned_node in C_set:
    # For each conditioned node, find all of its ancestors in G_work and add them, together with the node itself, to nodes_of_interest.
        nodes_of_interest.update(nx.ancestors(G_work, conditioned_node))
        nodes_of_interest.add(conditioned_node)
    # Construct the local graph containing only these ancestral nodes.
    G_0 = G_work.subgraph(nodes_of_interest).copy()
     # Extract all nodes in nodes_of_interest in reverse beta order, from right to left, as the processing order.
    vertices = [node for node in reversed(beta_list) if node in nodes_of_interest]
    E_beta_R: list[Edge] = []
    # Complete parent sets node by node.
    for current_node in vertices[:-2]:# The final two nodes do not need to be processed.
        parents = list(G_0.predecessors(current_node))
        for x, y in combinations(parents, 2):# Enumerate all pairs of parent nodes.
            if G_0.has_edge(x, y) or G_0.has_edge(y, x):
                continue # Skip the pair if an edge already exists between them.
            if beta_position[x] < beta_position[y]:
                new_edge = (x, y)
            else:
                new_edge = (y, x)
            G_0.add_edge(*new_edge)# Add the directed edge according to the order beta.
            E_beta_R.append(new_edge)
        G_0.remove_node(current_node)# Remove the current node after it has been processed.
   # Construct the final graph.
    R_nodes = [node for node in G.nodes if node not in C_set]
    G_beta_R = G.subgraph(R_nodes).copy()
    G_beta_R.add_edges_from(E_beta_R)
    return E_beta_R, G_beta_R

# Run ROS without timing.
# Directly call the core algorithm and package the result as a ROSResult object.
def minimal_fill_edge_set(G: nx.DiGraph, beta: Sequence[Node], C: Iterable[Node]) -> ROSResult:
    fill_edges, G_beta_R = _ros_algorithm_core(G=G, beta=beta, C=C)
    return ROSResult(E_beta_R=tuple(fill_edges), G_beta_R=G_beta_R)

# Timed ROS execution; elapsed time is measured in nanoseconds.
def _timed_ros_once(G: nx.DiGraph, beta: Sequence[Node], C: Iterable[Node]) -> tuple[ROSResult, int]:
    start = time.perf_counter_ns()   # Start timing.
    fill_edges, G_beta_R = _ros_algorithm_core(G=G, beta=beta, C=C)
    end = time.perf_counter_ns() # End timing.
    result = ROSResult(E_beta_R=tuple(fill_edges), G_beta_R=G_beta_R)
    return result, end - start

## Loading the Structures of Six Standard Real-World Networks


In [5]:
# Define the expected node counts for the six standard Bayesian networks to verify that the BIF files are loaded correctly.
EXPECTED_NODE_COUNTS = {"Asia": 8, "Insurance": 27, "Alarm": 37, "Win95pts": 76, "Andes": 223, "Munin": 1041}

# Clean variable names read from BIF files by removing leading/trailing whitespace and surrounding quotation marks.
def _clean_bif_name(raw_name: str) -> str:
    name = raw_name.strip()
    if len(name) >= 2 and name[0] == name[-1] and name[0] in {"'", '"'}:
        name = name[1:-1]
    return name.strip()

# If pgmpy is unavailable, use regular expressions to extract nodes and parent-child relationships from the BIF file.
def _load_bif_structure_fallback(path: Path) -> nx.DiGraph:
    text = path.read_text(encoding="utf-8", errors="ignore")
    text = re.sub(r"/\*.*?\*/", "", text, flags=re.S)
    text = re.sub(r"//.*?$", "", text, flags=re.M)
    G = nx.DiGraph(name=path.stem)
    variable_pattern = re.compile(r"\bvariable\s+(\"[^\"]+\"|'[^']+'|[^\s\{]+)\s*\{", flags=re.I)
    for match in variable_pattern.finditer(text):
        node = _clean_bif_name(match.group(1))
        if node:
            G.add_node(node)
    probability_pattern = re.compile(r"\bprobability\s*\(\s*(.*?)\s*\)\s*\{", flags=re.I | re.S)
    for match in probability_pattern.finditer(text):
        header = match.group(1).strip()
        if "|" in header:
            child_text, parent_text = header.split("|", 1)
            parents = [_clean_bif_name(item) for item in parent_text.split(",") if _clean_bif_name(item)]
        else:
            child_text = header
            parents = []
        child = _clean_bif_name(child_text)
        if not child:
            continue
        G.add_node(child)
        for parent in parents:
            G.add_edge(parent, child)
    if G.number_of_nodes() == 0:
        raise ValueError(f"未能从 BIF 文件读取任何节点：{path}")
    return G

# Load a standard-network DAG from a BIF file.
def load_bif_dag(path: str | Path) -> nx.DiGraph:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"BIF 文件不存在：{path}")
    try:
        from pgmpy.readwrite import BIFReader
    except ImportError:
        G = _load_bif_structure_fallback(path)
    else:
        model = BIFReader(str(path)).get_model()
        G = nx.DiGraph(name=path.stem)
        G.add_nodes_from(model.nodes())
        G.add_edges_from(model.edges())
    _validate_dag(G)
    return G

# Load the six standard networks in batch.
def load_standard_networks(data_dir: str | Path, network_files: Mapping[str, str], strict_node_count: bool = True) -> dict[str, nx.DiGraph]:
    data_dir = Path(data_dir)
    graphs: dict[str, nx.DiGraph] = {}
    for network, filename in network_files.items():
        G = load_bif_dag(data_dir / filename)
        G.name = network
        expected_n = EXPECTED_NODE_COUNTS.get(network)
        if strict_node_count and expected_n is not None and G.number_of_nodes() != expected_n:
            raise ValueError(f"{network} 节点数不正确：读取到 {G.number_of_nodes()}，预期 {expected_n}。")
        graphs[network] = G
    return graphs

## Experiment A: Structural Consistency on Standard Networks
Each of the six networks uses 30 three-node conditioning sets and 4 fixed topological orders, giving a total of
$6\times30\times4=720$ paired instances.


In [6]:
# Define an immutable dataclass to store the parameters of the consistency experiment.
@dataclass(frozen=True)
class StandardConsistencyConfig:
    c_size: int = 3  # Conditioning-set size (|C|).
    condition_sets_per_network: int = 30  # Number of randomly generated conditioning sets for each network.
    n_fixed_orders: int = 4  # Number of fixed topological orders generated for each network.
    condition_base_seed: int = 20260729  # Base random seed used to generate conditioning sets.
    order_base_seed: int = 20260812  # Base random seed used to generate topological orders.
    stop_on_mismatch: bool = False  # Whether to stop the entire experiment immediately if FGCR and ROS outputs differ.

# Validate the numerical configuration; all parameters must be positive integers.
def _validate_consistency_config(config: StandardConsistencyConfig) -> None:
    if config.c_size <= 0:
        raise ValueError("c_size 必须为正整数。")
    if config.condition_sets_per_network <= 0:
        raise ValueError("condition_sets_per_network 必须为正整数。")
    if config.n_fixed_orders <= 0:
        raise ValueError("n_fixed_orders 必须为正整数。")

# Run one consistency test by comparing FGCR and ROS outputs under the same (G, C, beta).
def run_one_consistency_case(network: str, G: nx.DiGraph, condition_id: int, order_id: int, C: Sequence[Node], beta: Sequence[Node], condition_seed: int, beta_seed: int) -> dict[str, object]:
    try: # Execute the FGCR and ROS algorithms.
        fgcr_result = run_fgcr(G=G, C=C, beta=beta)
        ros_result = minimal_fill_edge_set(G=G, beta=beta, C=C)
        G_R = G.subgraph(fgcr_result.orders.alpha_R).copy()
        fgcr_fill = set(fgcr_result.H_R.edges()) - set(G_R.edges())
        ros_fill = set(ros_result.E_beta_R)
        fill_equal = fgcr_fill == ros_fill
        graph_equal = set(fgcr_result.H_R.nodes()) == set(ros_result.G_beta_R.nodes()) and set(fgcr_result.H_R.edges()) == set(ros_result.G_beta_R.edges())
        #Construct the result dictionary containing network information, seeds, node/edge counts, encoded conditioning sets and topological orders, various counts, comparison results, and encoded edge sets for debugging.
        return {
            "network": network,
            "condition_id": condition_id,
            "order_id": order_id,
            "condition_seed": condition_seed,
            "beta_seed": beta_seed,
            "n_nodes": G.number_of_nodes(),
            "n_edges": G.number_of_edges(),
            "c_size": len(C),
            "C": _encode_nodes(C),
            "beta": _encode_order(beta),
            "alpha_C": _encode_order(fgcr_result.orders.alpha_C),
            "alpha_R": _encode_order(fgcr_result.orders.alpha_R),
            "alpha": _encode_order(fgcr_result.orders.alpha),
            "FGCR_fill_count": len(fgcr_fill),
            "ROS_fill_count": len(ros_fill),
            "FGCR_output_edge_count": fgcr_result.H_R.number_of_edges(),
            "ROS_output_edge_count": ros_result.G_beta_R.number_of_edges(),
            "fill_equal": fill_equal,
            "graph_equal": graph_equal,
            "both_equal": fill_equal and graph_equal,
            "FGCR_fill_edges": _encode_edges(fgcr_fill),
            "ROS_fill_edges": _encode_edges(ros_fill),
            "FGCR_output_edges": _encode_edges(fgcr_result.H_R.edges()),
            "ROS_output_edges": _encode_edges(ros_result.G_beta_R.edges()),
            "status": "ok",
            "error_type": "",
            "error_message": "",
        }
    #Catch any exception, return the error information, set fill_equal, graph_equal, and both_equal to False, and leave encoded output fields empty.
    except Exception as exc:
        return {
            "network": network,
            "condition_id": condition_id,
            "order_id": order_id,
            "condition_seed": condition_seed,
            "beta_seed": beta_seed,
            "n_nodes": G.number_of_nodes(),
            "n_edges": G.number_of_edges(),
            "c_size": len(C),
            "C": _encode_nodes(C),
            "beta": _encode_order(beta),
            "alpha_C": "",
            "alpha_R": "",
            "alpha": "",
            "FGCR_fill_count": None,
            "ROS_fill_count": None,
            "FGCR_output_edge_count": None,
            "ROS_output_edge_count": None,
            "fill_equal": False,
            "graph_equal": False,
            "both_equal": False,
            "FGCR_fill_edges": "",
            "ROS_fill_edges": "",
            "FGCR_output_edges": "",
            "ROS_output_edges": "",
            "status": "error",
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }

# Summary function grouped by network.
def _summarize_consistency(details: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for network, group in details.groupby("network", sort=False):
        rows.append({
            "network": network,
            "n_nodes": int(group["n_nodes"].iloc[0]),
            "n_edges": int(group["n_edges"].iloc[0]),
            "c_size": int(group["c_size"].iloc[0]),
            "n_instances": len(group),
            "n_condition_sets": group["condition_id"].nunique(),
            "n_orders": group["order_id"].nunique(),
            "fill_equal_rate": float(group["fill_equal"].mean()),
            "graph_equal_rate": float(group["graph_equal"].mean()),
            "both_equal_rate": float(group["both_equal"].mean()),
             # Count the number of instances with errors or inconsistencies.
            "mismatch_or_error_count": int(((group["status"] != "ok") | (~group["both_equal"])).sum()),
        })
    return pd.DataFrame(rows)

# Experiment function.
def run_standard_consistency_experiment(network_graphs: Mapping[str, nx.DiGraph], config: StandardConsistencyConfig = StandardConsistencyConfig()) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    _validate_consistency_config(config)
    rows: list[dict[str, object]] = []
    order_rows: list[dict[str, object]] = []
      # Iterate over each network.
    for network_index, (network, G) in enumerate(network_graphs.items(), start=1):
        _validate_dag(G)
        condition_seed = config.condition_base_seed + network_index * 1_000_000 + 101
        order_seed_base = config.order_base_seed + network_index * 1_000_000
        condition_sets = sample_random_condition_sets(G=G, n_sets=config.condition_sets_per_network, c_size=config.c_size, seed=condition_seed)
        fixed_orders = generate_fixed_random_topological_orders(G=G, n_orders=config.n_fixed_orders, seed_base=order_seed_base)
        for order_info in fixed_orders:
            order_rows.append({"network": network, "order_id": order_info["order_id"], "beta_seed": order_info["beta_seed"], "beta": _encode_order(order_info["beta"])})
        for condition_id, C in enumerate(condition_sets, start=1):
            # Double loop: combine each conditioning set with each topological order and call run_one_consistency_case once for every pair.
            for order_info in fixed_orders:
                row = run_one_consistency_case(
                    network=network, G=G, condition_id=condition_id, order_id=int(order_info["order_id"]),
                    C=C, beta=order_info["beta"], condition_seed=condition_seed, beta_seed=int(order_info["beta_seed"])
                )
                rows.append(row)
                if config.stop_on_mismatch and not row["both_equal"]:
                    #If configured to stop on mismatch, raise an exception immediately.
                    raise AssertionError(f"{network}，条件集 {condition_id}，拓扑序 {order_info['order_id']} 未通过一致性检查。")
        print(f"[一致性][{network}] 完成 {len(condition_sets) * len(fixed_orders)} 个实例。")
    details = pd.DataFrame(rows)
    fixed_orders_df = pd.DataFrame(order_rows)
    mismatch_mask = (details["status"] != "ok") | (~details["both_equal"].fillna(False))
    mismatches = details.loc[mismatch_mask].copy()
    summary = _summarize_consistency(details)
    expected_instances = len(network_graphs) * config.condition_sets_per_network * config.n_fixed_orders
    if len(details) != expected_instances:
        #Check whether the total number of instances matches the expected count (6 networks × 30 conditioning sets × 4 topological orders = 720).
        raise AssertionError(f"一致性实验实例数错误：得到 {len(details)}，预期 {expected_instances}。")
    return details, summary, mismatches, fixed_orders_df

# Save multiple DataFrames and the experiment configuration to separate worksheets in a single Excel workbook.
def save_consistency_results(details: pd.DataFrame, summary: pd.DataFrame, mismatches: pd.DataFrame, fixed_orders: pd.DataFrame, config: StandardConsistencyConfig, output_file: str | Path) -> Path:
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    config_df = pd.DataFrame({"parameter": list(asdict(config).keys()), "value": list(asdict(config).values())})
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="network_summary", index=False)
        fixed_orders.to_excel(writer, sheet_name="fixed_orders", index=False)
        details.to_excel(writer, sheet_name="instance_results", index=False)
        mismatches.to_excel(writer, sheet_name="mismatches", index=False)
        config_df.to_excel(writer, sheet_name="config", index=False)
    return output_file

## Experiment B: Algorithm Runtime on Standard Networks
Each of the six networks uses 30 conditioning sets and only the first fixed topological order from the consistency experiment, giving a total of $6\times30=180$ timing instances. Each instance is warmed up twice and then formally repeated 10 times; the execution order of FGCR and ROS is alternated.

In [7]:
# Define the parameters for the runtime experiment.
@dataclass(frozen=True)
class RuntimeExperimentConfig:
    c_size: int = 3
    condition_sets_per_network: int = 30
    n_fixed_orders: int = 4  #  Number of distinct fixed topological orders generated for each standard network.
    warmup_repetitions: int = 3 # Independently warm up each fixed (G,C,beta) instance 3 times before formal timing.
    timing_repetitions: int = 10   # Formally repeat each fixed (G,C,beta) instance 10 times and take the median.
    condition_base_seed: int = 20260729
    order_base_seed: int = 20260812

def _validate_runtime_config(config: RuntimeExperimentConfig) -> None:# Validate the configuration.
    if config.c_size <= 0:
        raise ValueError("c_size 必须为正整数。")
    if config.condition_sets_per_network <= 0:
        raise ValueError("condition_sets_per_network 必须为正整数。")
    if config.n_fixed_orders <= 0:
        raise ValueError("n_fixed_orders 必须为正整数。")
    if config.warmup_repetitions < 0:
        raise ValueError("warmup_repetitions 不能为负数。")
    if config.timing_repetitions <= 0:
        raise ValueError("timing_repetitions 必须为正整数。")

# Run one timed FGCR-ROS pair with Python garbage collection (GC) disabled to reduce the influence of GC on timing.
def _run_pair_without_gc(G: nx.DiGraph, C: Sequence[Node], beta: Sequence[Node], fgcr_first: bool) -> tuple[int, int, int]:
    gc_was_enabled = gc.isenabled()
    if gc_was_enabled: # Record the current GC state and disable it if it is enabled.
        gc.disable()
    try:# Call _timed_fgcr_once and _timed_ros_once according to fgcr_first, retaining only their timing outputs and ignoring algorithm results with "_".
        if fgcr_first:
            _, algorithm1_ns, algorithm2_ns = _timed_fgcr_once(G=G, C=C, beta=beta)
            _, ros_total_ns = _timed_ros_once(G=G, beta=beta, C=C)
        else:
            _, ros_total_ns = _timed_ros_once(G=G, beta=beta, C=C)
            _, algorithm1_ns, algorithm2_ns = _timed_fgcr_once(G=G, C=C, beta=beta)
    finally:
        if gc_was_enabled: #Restore the original GC state.
            gc.enable()
    return algorithm1_ns, algorithm2_ns, ros_total_ns

# Warm-up function for stabilizing caches and other runtime effects. Perform warm-up separately for each complete (G, C, beta) instance.
def _warm_up_one_case(
    G: nx.DiGraph,
    C: Sequence[Node],
    beta: Sequence[Node],
    repetitions: int,
) -> None:
    for warmup_id in range(1, repetitions + 1):
        gc.collect()
        _run_pair_without_gc(
            G=G,
            C=C,
            beta=beta,
            fgcr_first=(warmup_id % 2 == 1),
        )

# Time one fixed conditioning-set and topological-order instance.
def run_one_runtime_case(network: str, G: nx.DiGraph, condition_id: int,order_id: int, C: Sequence[Node], beta: Sequence[Node], condition_seed: int, beta_seed: int, timing_repetitions: int) -> tuple[dict[str, object], list[dict[str, object]]]:
    algorithm1_times_ns: list[int] = []
    algorithm2_times_ns: list[int] = []
    fgcr_total_times_ns: list[int] = []
    ros_total_times_ns: list[int] = []
    repeat_rows: list[dict[str, object]] = []
    # For a fixed conditioning set C and topological order beta, repeat the experiment 10 times, with FGCR running first 5 times and ROS running first 5 times.
    for repeat_id in range(1, timing_repetitions + 1):
        gc.collect()
        fgcr_first = (condition_id + order_id+ repeat_id) % 2 == 0
        algorithm1_ns, algorithm2_ns, ros_total_ns = _run_pair_without_gc(G=G, C=C, beta=beta, fgcr_first=fgcr_first)
        fgcr_total_ns = algorithm1_ns + algorithm2_ns
        algorithm1_times_ns.append(algorithm1_ns)
        algorithm2_times_ns.append(algorithm2_ns)
        fgcr_total_times_ns.append(fgcr_total_ns)
        ros_total_times_ns.append(ros_total_ns)
        repeat_rows.append({
            "network": network,
            "condition_id": condition_id,
            "order_id": order_id,
            "condition_seed": condition_seed,
            "beta_seed": beta_seed,
            "repeat_id": repeat_id,
            "run_order": "FGCR_then_ROS" if fgcr_first else "ROS_then_FGCR",
            "FGCR_algorithm1_ms": algorithm1_ns / 1_000_000,
            "FGCR_algorithm2_ms": algorithm2_ns / 1_000_000,
            "FGCR_total_ms": fgcr_total_ns / 1_000_000,
            "ROS_total_ms": ros_total_ns / 1_000_000,
        })
    #After all repetitions, take the median of each timing quantity using statistics.median as the representative value for the instance.
    algorithm1_median_ns = float(median(algorithm1_times_ns))
    algorithm2_median_ns = float(median(algorithm2_times_ns))
    fgcr_total_median_ns = float(median(fgcr_total_times_ns))
    ros_total_median_ns = float(median(ros_total_times_ns))
    # Store the representative runtime for the current (G, C, beta) instance.
    condition_row = {
        "network": network,
        "condition_id": condition_id,
        "order_id": order_id,
        "condition_seed": condition_seed,
        "beta_seed": beta_seed,
        "n_nodes": G.number_of_nodes(),
        "n_edges": G.number_of_edges(),
        "c_size": len(C),
        "C": _encode_nodes(C),
        "timing_repetitions": timing_repetitions,
        "FGCR_algorithm1_median_ms": algorithm1_median_ns / 1_000_000,
        "FGCR_algorithm2_median_ms": algorithm2_median_ns / 1_000_000,
        "FGCR_total_median_ms": fgcr_total_median_ns / 1_000_000,
        "ROS_total_median_ms": ros_total_median_ns / 1_000_000,
        "FGCR_to_ROS_ratio": fgcr_total_median_ns / ros_total_median_ns,  
        #  Runtime ratio: values below 1 indicate that FGCR is faster, whereas values above 1 indicate that ROS is faster.
    }
    return condition_row, repeat_rows

#Summary function.
def _summarize_runtime(details: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    #Group by network and average the instance-level median runtimes.
    for network, group in details.groupby("network", sort=False):
        fgcr_algorithm1_mean = float(group["FGCR_algorithm1_median_ms"].mean())
        fgcr_algorithm2_mean = float(group["FGCR_algorithm2_median_ms"].mean())
        fgcr_total_mean = float(group["FGCR_total_median_ms"].mean())
        ros_total_mean = float(group["ROS_total_median_ms"].mean())
        rows.append({
            "network": network,
            "n_nodes": int(group["n_nodes"].iloc[0]),
            "n_edges": int(group["n_edges"].iloc[0]),
            "c_size": int(group["c_size"].iloc[0]),
            "n_instances": len(group),# Total number of conditioning-set × topological-order instances.
            "n_condition_sets": group["condition_id"].nunique(), # Number of distinct conditioning sets; expected to be 30.
            "n_orders": group["order_id"].nunique(), #  Number of distinct topological orders; expected to be 4.
            "n_unique_condition_sets": group["C"].nunique(),
            "FGCR_algorithm1_mean_ms": fgcr_algorithm1_mean,
            "FGCR_algorithm2_mean_ms": fgcr_algorithm2_mean,
            "FGCR_total_mean_ms": fgcr_total_mean,
            "ROS_total_mean_ms": ros_total_mean,
            "FGCR_to_ROS_ratio": fgcr_total_mean / ros_total_mean,
        })
    return pd.DataFrame(rows)

# Summarize runtimes by network and topological order.
def _summarize_runtime_by_order(details: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for (network, order_id), group in details.groupby(["network", "order_id"], sort=False):
        fgcr_algorithm1_mean = float(group["FGCR_algorithm1_median_ms"].mean())
        fgcr_algorithm2_mean = float(group["FGCR_algorithm2_median_ms"].mean())
        fgcr_total_mean = float(group["FGCR_total_median_ms"].mean())
        ros_total_mean = float(group["ROS_total_median_ms"].mean())
        rows.append({
            "network": network,
            "order_id": int(order_id),
            "beta_seed": int(group["beta_seed"].iloc[0]),
            "n_nodes": int(group["n_nodes"].iloc[0]),
            "n_edges": int(group["n_edges"].iloc[0]),
            "c_size": int(group["c_size"].iloc[0]),
            "n_condition_sets": int(group["condition_id"].nunique()),
            "FGCR_algorithm1_mean_ms": fgcr_algorithm1_mean,
            "FGCR_algorithm2_mean_ms": fgcr_algorithm2_mean,
            "FGCR_total_mean_ms": fgcr_total_mean,
            "ROS_total_mean_ms": ros_total_mean,
            "FGCR_to_ROS_ratio": fgcr_total_mean / ros_total_mean,
        })
    return pd.DataFrame(rows)

#Experiment function.
def run_runtime_experiment(network_graphs: Mapping[str, nx.DiGraph], config: RuntimeExperimentConfig = RuntimeExperimentConfig()) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    _validate_runtime_config(config)
    condition_rows: list[dict[str, object]] = []
    repeat_rows: list[dict[str, object]] = []
    fixed_input_rows: list[dict[str, object]] = []
    for network_index, (network, G) in enumerate(network_graphs.items(), start=1):
        _validate_dag(G)
         # Generate fixed seeds, conditioning sets, and topological orders.
        condition_seed = config.condition_base_seed + network_index * 1_000_000 + 101
        # Construct the seed starting point for the four fixed topological orders of the current network.
        order_seed_base = (config.order_base_seed+ network_index * 1_000_000)
         # Generate 30 distinct fixed conditioning sets.
        condition_sets = sample_random_condition_sets(G=G, n_sets=config.condition_sets_per_network, c_size=config.c_size, seed=condition_seed)
        fixed_orders = generate_fixed_random_topological_orders( G=G, n_orders=config.n_fixed_orders, seed_base=order_seed_base, )
         # Run the experiment separately for each of the four fixed topological orders.
        for order_info in fixed_orders:
            order_id = int( order_info["order_id"] )
            beta_seed = int( order_info["beta_seed"])
            beta = tuple(order_info["beta"])
            # Store the current fixed topological order and experimental input information.
            fixed_input_rows.append({
                "network": network,
                "order_id": order_id,
                "n_nodes": G.number_of_nodes(),
                "n_edges": G.number_of_edges(),
                "condition_seed": condition_seed,
                "beta_seed": beta_seed,
                "c_size": config.c_size,
                "n_condition_sets": len(condition_sets),
                "n_unique_condition_sets": len(set(condition_sets)),
                "beta": _encode_order(beta),
            })
             # Treat each fixed (G, C, beta) combination as an independent timing instance.
            for condition_id, C in enumerate(condition_sets, start=1):
                # Loop over each conditioning set, call run_one_runtime_case, and collect the resulting timing records.
                _warm_up_one_case(G=G,C=C, beta=beta,repetitions=config.warmup_repetitions, )
                condition_row, current_repeat_rows = run_one_runtime_case(
                    network=network, G=G, condition_id=condition_id,order_id=order_id, C=C, beta=beta,
                    condition_seed=condition_seed, beta_seed=beta_seed,
                    timing_repetitions=config.timing_repetitions
                )
                condition_rows.append(condition_row)
                repeat_rows.extend(current_repeat_rows)
        current_details = pd.DataFrame(row for row in condition_rows if row["network"] == network)
        current_summary = _summarize_runtime(current_details).iloc[0]
        completed_instances = ( len(condition_sets) * len(fixed_orders) )
        print(f"[计时][{network}] " f"完成 {completed_instances} 个" f"(条件集, 拓扑序)实例；"
            f"FGCR="f"{current_summary['FGCR_total_mean_ms']:.6f} ms，"
            f"ROS="f"{current_summary['ROS_total_mean_ms']:.6f} ms。")
    details = pd.DataFrame(condition_rows)
    repeat_times = pd.DataFrame(repeat_rows)
    fixed_inputs = pd.DataFrame(fixed_input_rows)
    summary = _summarize_runtime(details)
    order_summary = (_summarize_runtime_by_order(details))
   
    expected_instances = len(network_graphs) * config.condition_sets_per_network* config.n_fixed_orders
    if len(details) != expected_instances:
        raise AssertionError(f"计时实验实例数错误：得到 {len(details)}，预期 {expected_instances}。")
    
    expected_repeat_rows = (expected_instances* config.timing_repetitions )
    if len(repeat_times) != expected_repeat_rows:
        raise AssertionError(  "原始重复计时行数错误："f"得到 {len(repeat_times)}，"f"预期 {expected_repeat_rows}。")
    
    expected_fixed_inputs = (len(network_graphs)* config.n_fixed_orders)
    if len(fixed_inputs) != expected_fixed_inputs:
        raise AssertionError( "固定拓扑序记录数错误："f"得到 {len(fixed_inputs)}，"  f"预期 {expected_fixed_inputs}。")
    return summary, order_summary,details, repeat_times, fixed_inputs

# Save the results.
def save_runtime_results(summary: pd.DataFrame, order_summary: pd.DataFrame,details: pd.DataFrame, repeat_times: pd.DataFrame, fixed_inputs: pd.DataFrame, config: RuntimeExperimentConfig, output_file: str | Path) -> Path:
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    
    config_dict = asdict(config)
    config_df = pd.DataFrame({"parameter": list(asdict(config).keys()), "value": list(asdict(config).values())})
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="network_summary", index=False) 
        
        order_summary.to_excel( writer,sheet_name="order_summary",index=False,)
        details.to_excel(writer, sheet_name="case_details", index=False)  
        repeat_times.to_excel(writer, sheet_name="raw_repeat_times", index=False) 
        fixed_inputs.to_excel(writer, sheet_name="fixed_inputs", index=False) 
        config_df.to_excel(writer, sheet_name="config", index=False)  
    return output_file

## Formal Configuration and Execution Entry Point


In [8]:
STANDARD_DATA_DIR = Path(r"D:\jupyternoteboook\实验\计时实验\data")# Define the path to the data directory.

# Define the mapping from network names to BIF filenames for the six standard networks.
STANDARD_NETWORK_FILES = {"Asia": "asia.bif", "Insurance": "insurance.bif", "Alarm": "alarm.bif", "Win95pts": "win95pts.bif", "Andes": "andes.bif", "Munin": "munin.bif"}

# Create the configuration object for the structural-consistency experiment.
STANDARD_CONSISTENCY_CONFIG = StandardConsistencyConfig(c_size=3, condition_sets_per_network=30, n_fixed_orders=4, condition_base_seed=20260729, order_base_seed=20260812, stop_on_mismatch=False)

# Create the configuration object for the runtime experiment.
STANDARD_RUNTIME_CONFIG = RuntimeExperimentConfig(c_size=3, condition_sets_per_network=30,n_fixed_orders = 4, warmup_repetitions=3, timing_repetitions=10, condition_base_seed=20260729, order_base_seed=20260812)

STANDARD_RESULT_DIR = Path("./results_standard_networks")# Root directory for result output.
STANDARD_CONSISTENCY_RESULT_FILE = STANDARD_RESULT_DIR / "FGCR_ROS_consistency.xlsx"
STANDARD_RUNTIME_RESULT_FILE = STANDARD_RESULT_DIR / "FGCR_ROS_runtime.xlsx"

# Reload the six BIF files -> run 720 instances -> save to Excel -> print summary.
# Independently run the 720-instance output-consistency experiment on the standard networks.
def run_standard_consistency_only(data_dir: str | Path = STANDARD_DATA_DIR, network_files: Mapping[str, str] = STANDARD_NETWORK_FILES, config: StandardConsistencyConfig = STANDARD_CONSISTENCY_CONFIG, output_file: str | Path = STANDARD_CONSISTENCY_RESULT_FILE) -> dict[str, object]:
    network_graphs = load_standard_networks(data_dir=data_dir, network_files=network_files, strict_node_count=True)
    details, summary, mismatches, fixed_orders = run_standard_consistency_experiment(network_graphs=network_graphs, config=config)
    saved_path = save_consistency_results(details=details, summary=summary, mismatches=mismatches, fixed_orders=fixed_orders, config=config, output_file=output_file)
    print(summary.to_string(index=False))
    print(f"不一致或报错实例数：{len(mismatches)}")
    print(f"结果文件：{saved_path.resolve()}")
    return {"summary": summary, "details": details, "mismatches": mismatches, "fixed_orders": fixed_orders, "output_file": saved_path}

# Reload the six BIF files -> run 720 timing instances -> save to Excel -> print summary.
# Independently run the 720-instance pure algorithm timing experiment on the standard networks.
def run_standard_runtime_only(data_dir: str | Path = STANDARD_DATA_DIR, network_files: Mapping[str, str] = STANDARD_NETWORK_FILES, config: RuntimeExperimentConfig = STANDARD_RUNTIME_CONFIG, output_file: str | Path = STANDARD_RUNTIME_RESULT_FILE) -> dict[str, object]:
    network_graphs = load_standard_networks(data_dir=data_dir, network_files=network_files, strict_node_count=True)
    (summary,order_summary,details,repeat_times,fixed_inputs,)= run_runtime_experiment(network_graphs = network_graphs,config = config,)
    saved_path = save_runtime_results(summary=summary,order_summary = order_summary, details=details, repeat_times=repeat_times, fixed_inputs=fixed_inputs, config=config, output_file=output_file)
    # Print the network-level summary.
    print("\n网络级计时汇总：")
    print(summary.to_string( index=False ))
    # Print the network-by-topological-order summary.
    print("\n不同拓扑序计时汇总：")
    print( order_summary.to_string( index=False) )
    print( f"\n正式计时实例数：{len(details)}" )
    print("原始重复计时记录数：" f"{len(repeat_times)}")
    print( f"结果文件：{saved_path.resolve()}")
    return { "summary": summary, "order_summary": order_summary, "details": details, "repeat_times": repeat_times, "fixed_inputs": fixed_inputs, "output_file": saved_path,}
# Run the two independent experiments separately.
consistency_result = run_standard_consistency_only()
runtime_result = run_standard_runtime_only()

D:\Anaconda\envs\new_env_1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[一致性][Asia] 完成 120 个实例。
[一致性][Insurance] 完成 120 个实例。
[一致性][Alarm] 完成 120 个实例。
[一致性][Win95pts] 完成 120 个实例。
[一致性][Andes] 完成 120 个实例。
[一致性][Munin] 完成 120 个实例。


C:\Users\28936\AppData\Local\Temp\ipykernel_12716\1705625423.py:161: UserWarning: Cell contents too long (89100), truncated to 32767 characters
  details.to_excel(writer, sheet_name="instance_results", index=False)
C:\Users\28936\AppData\Local\Temp\ipykernel_12716\1705625423.py:161: UserWarning: Cell contents too long (89989), truncated to 32767 characters
  details.to_excel(writer, sheet_name="instance_results", index=False)
C:\Users\28936\AppData\Local\Temp\ipykernel_12716\1705625423.py:161: UserWarning: Cell contents too long (89547), truncated to 32767 characters
  details.to_excel(writer, sheet_name="instance_results", index=False)
C:\Users\28936\AppData\Local\Temp\ipykernel_12716\1705625423.py:161: UserWarning: Cell contents too long (91517), truncated to 32767 characters
  details.to_excel(writer, sheet_name="instance_results", index=False)
C:\Users\28936\AppData\Local\Temp\ipykernel_12716\1705625423.py:161: UserWarning: Cell contents too long (49181), truncated to 32767 charact

  network  n_nodes  n_edges  c_size  n_instances  n_condition_sets  n_orders  fill_equal_rate  graph_equal_rate  both_equal_rate  mismatch_or_error_count
     Asia        8        8       3          120                30         4              1.0               1.0              1.0                        0
Insurance       27       52       3          120                30         4              1.0               1.0              1.0                        0
    Alarm       37       46       3          120                30         4              1.0               1.0              1.0                        0
 Win95pts       76      112       3          120                30         4              1.0               1.0              1.0                        0
    Andes      223      338       3          120                30         4              1.0               1.0              1.0                        0
    Munin     1041     1397       3          120                30         4